In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import cv2

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import json
from collections import deque

import tensorflow as tf

In [2]:
# MediaPipe hand connections 
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),        # thumb
    (0,5),(5,6),(6,7),(7,8),        # index
    (0,9),(9,10),(10,11),(11,12),   # middle  
    (0,13),(13,14),(14,15),(15,16), # ring    
    (0,17),(17,18),(18,19),(19,20), # pinky
    (5,9),(9,13),(13,17)            # palm
]

In [3]:
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)
print("mediapipe:", mp.__version__)

numpy: 2.4.2
opencv: 4.13.0
mediapipe: 0.10.32


#### Mediapipe Hand Landmarks in Real Time

In [ ]:
model_path = os.path.join("mp", "hand_landmarker.task")
if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Could not find {model_path} in {os.getcwd()}. Put the model next to this notebook or use an absolute path."
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    running_mode=vision.RunningMode.IMAGE,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    )

detector = vision.HandLandmarker.create_from_options(options)


cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions and close other apps using the camera.")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    h, w, _ = frame.shape
    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            points = []
            for lm in hand_landmarks:
                x = int(lm.x * w)
                y = int(lm.y * h)
                points.append((x, y))
                cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

            for start_idx, end_idx in HAND_CONNECTIONS:
                x1, y1 = points[start_idx]
                x2, y2 = points[end_idx]
                cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

    cv2.putText(frame, "Press q to quit", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.imshow("MediaPipe Hand Landmarks", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

### Model testing 
1. Prep (load model + metadata)
2. Helpers and stability config
3. Real-time loop 

In [18]:
# prep (metadata + TFLite model/interpreter)
ARTIFACTS_DIR = "artifacts"
META_PATH = os.path.join(ARTIFACTS_DIR, "lstm_dataset_meta.json")
TFLITE_CANDIDATE = os.path.join(ARTIFACTS_DIR, "tf_model_dir", "model_float32.tflite")
TF_MODEL_DIR = os.path.join(ARTIFACTS_DIR, "tf_model_dir")
HAND_LANDMARKER_PATH = os.path.join("mp", "hand_landmarker.task")

if not os.path.isfile(HAND_LANDMARKER_PATH):
    raise FileNotFoundError(f"Missing MediaPipe model: {HAND_LANDMARKER_PATH}")
if not os.path.isfile(META_PATH):
    raise FileNotFoundError(f"Missing metadata file: {META_PATH}")

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

seq_len = int(meta["seq_len"])
num_features = int(meta["num_features"])
class_names = list(meta["class_names"])
num_classes = int(meta["num_classes"])
scaler_mean = np.asarray(meta["scaler_mean"], dtype=np.float32)
scaler_scale = np.asarray(meta["scaler_scale"], dtype=np.float32)

if scaler_mean.shape[0] != num_features or scaler_scale.shape[0] != num_features:
    raise ValueError("Scaler arrays in metadata do not match num_features")

print(f"seq_len={seq_len}, num_features={num_features}, num_classes={num_classes}")

tflite_model_path = None
if os.path.isfile(TFLITE_CANDIDATE):
    tflite_model_path = TFLITE_CANDIDATE
else:
    tflite_in_dir = []
    if os.path.isdir(TF_MODEL_DIR):
        for root, _, files in os.walk(TF_MODEL_DIR):
            for filename in files:
                if filename.lower().endswith(".tflite"):
                    tflite_in_dir.append(os.path.join(root, filename))

    if tflite_in_dir:
        tflite_model_path = tflite_in_dir[0]
    elif os.path.isfile(os.path.join(TF_MODEL_DIR, "saved_model.pb")):
        print("No .tflite found in tf_model_dir; converting SavedModel to TFLite...")
        converter = tf.lite.TFLiteConverter.from_saved_model(TF_MODEL_DIR)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()
        tflite_model_path = os.path.join(ARTIFACTS_DIR, "lstm_model_from_savedmodel.tflite")
        with open(tflite_model_path, "wb") as f_out:
            f_out.write(tflite_model)

if tflite_model_path is None:
    raise FileNotFoundError(
        "Could not find a TFLite model. Place one at artifacts/lstm_model.tflite or inside artifacts/tf_model_dir/."
    )

print(f"Using TFLite model: {tflite_model_path}")
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Model input shape:", input_details[0]["shape"], "shape_signature:", input_details[0].get("shape_signature"))
print("Model output shape:", output_details[0]["shape"], "shape_signature:", output_details[0].get("shape_signature"))

def _shape_compatible(expected_shape, actual_shape):
    if len(expected_shape) != len(actual_shape):
        return False
    for exp_dim, act_dim in zip(expected_shape, actual_shape):
        if exp_dim in (-1, 0):
            continue
        if int(exp_dim) != int(act_dim):
            return False
    return True

def prepare_tflite_input(seq_array, input_detail):
    seq_feat = seq_array.astype(np.float32)

    candidates = [
        seq_feat[np.newaxis, :, :],
        np.transpose(seq_feat, (1, 0))[np.newaxis, :, :],
        seq_feat.reshape(1, -1),
        seq_feat.reshape(-1),
        seq_feat,
        np.transpose(seq_feat, (1, 0)),
        seq_feat[-1][np.newaxis, :],
        seq_feat[-1],
    ]

    shape = tuple(int(x) for x in input_detail["shape"])
    shape_sig = tuple(int(x) for x in input_detail.get("shape_signature", input_detail["shape"]))

    for cand in candidates:
        cand_shape = tuple(cand.shape)
        if _shape_compatible(shape, cand_shape) or _shape_compatible(shape_sig, cand_shape):
            return cand.astype(np.float32)

    raise ValueError(
        f"No compatible input layout found. Model expects shape={shape}, shape_signature={shape_sig}, "
        f"but generated sequence shape is {seq_feat.shape}."
    )

seq_len=10, num_features=126, num_classes=57
Using TFLite model: artifacts\tf_model_dir\model_float32.tflite
Model input shape: [  1 126  10] shape_signature: [ -1 126  10]
Model output shape: [ 1 57] shape_signature: [-1 57]


In [19]:
# MediaPipe helpers + stability config
base_options = python.BaseOptions(model_asset_path=HAND_LANDMARKER_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    running_mode=vision.RunningMode.IMAGE,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
detector = vision.HandLandmarker.create_from_options(options)

def get_handedness_label(result, hand_idx):
    try:
        handed = result.handedness[hand_idx]
        category = handed[0] if isinstance(handed, list) and len(handed) > 0 else handed
        label = getattr(category, "category_name", None) or getattr(category, "display_name", None)
        if isinstance(label, str):
            label = label.lower()
            if "left" in label:
                return "left"
            if "right" in label:
                return "right"
    except Exception:
        pass
    return None

def extract_ordered_frame_features(result, num_landmarks=21):
    detected_hands = list(getattr(result, "hand_landmarks", []) or [])
    ordered = {"left": None, "right": None}
    unknown = []

    for i, hand_lms in enumerate(detected_hands):
        label = get_handedness_label(result, i)
        if label in ordered and ordered[label] is None:
            ordered[label] = hand_lms
        else:
            unknown.append(hand_lms)

    if unknown:
        unknown = sorted(
            unknown,
            key=lambda hand: hand[0].x if hand and len(hand) > 0 else 0.5,
        )
        for hand_lms in unknown:
            if ordered["left"] is None:
                ordered["left"] = hand_lms
            elif ordered["right"] is None:
                ordered["right"] = hand_lms

    values = []
    for side in ("left", "right"):
        hand_lms = ordered[side]
        if hand_lms is None:
            values.extend([0.0] * num_landmarks * 3)
            continue

        for lm in hand_lms[:num_landmarks]:
            values.extend([lm.x, lm.y, lm.z])

        missing_pts = num_landmarks - len(hand_lms[:num_landmarks])
        if missing_pts > 0:
            values.extend([0.0] * missing_pts * 3)

    return np.asarray(values, dtype=np.float32)

# Stability controls
EMA_ALPHA = 0.2
SWITCH_CONF_THRESHOLD = 0.55
MIN_STREAK_TO_SWITCH = 8
FRAME_SKIP = 2

print(
    f"Stability settings -> EMA_ALPHA={EMA_ALPHA}, "
    f"SWITCH_CONF_THRESHOLD={SWITCH_CONF_THRESHOLD}, "
    f"MIN_STREAK_TO_SWITCH={MIN_STREAK_TO_SWITCH}, FRAME_SKIP={FRAME_SKIP}"
)

Stability settings -> EMA_ALPHA=0.2, SWITCH_CONF_THRESHOLD=0.55, MIN_STREAK_TO_SWITCH=8, FRAME_SKIP=2


In [20]:
# real-time loop 
sequence_buffer = deque(maxlen=seq_len)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions and close other apps using the camera.")

display_label = "..."
display_conf = 0.0
pending_label = None
pending_streak = 0
ema_probs = None
frame_count = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)

        h, w, _ = frame.shape
        hands_present = bool(result.hand_landmarks)

        if hands_present:
            for hand_landmarks in result.hand_landmarks:
                points = []
                for lm in hand_landmarks:
                    x = int(lm.x * w)
                    y = int(lm.y * h)
                    points.append((x, y))
                    cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

                for start_idx, end_idx in HAND_CONNECTIONS:
                    if start_idx < len(points) and end_idx < len(points):
                        x1, y1 = points[start_idx]
                        x2, y2 = points[end_idx]
                        cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

        if hands_present:
            frame_features = extract_ordered_frame_features(result)
            standardized = (frame_features - scaler_mean) / np.where(scaler_scale == 0, 1.0, scaler_scale)
            sequence_buffer.append(standardized.astype(np.float32))

            if len(sequence_buffer) == seq_len and frame_count % FRAME_SKIP == 0:
                seq_array = np.stack(sequence_buffer, axis=0).astype(np.float32)
                model_input = prepare_tflite_input(seq_array, input_details[0])
                interpreter.set_tensor(input_details[0]["index"], model_input)
                interpreter.invoke()

                raw_probs = interpreter.get_tensor(output_details[0]["index"])[0].astype(np.float32)

                if ema_probs is None:
                    ema_probs = raw_probs.copy()
                else:
                    ema_probs = EMA_ALPHA * raw_probs + (1.0 - EMA_ALPHA) * ema_probs

                smooth_idx = int(np.argmax(ema_probs))
                smooth_conf = float(ema_probs[smooth_idx])
                smooth_label = class_names[smooth_idx] if 0 <= smooth_idx < len(class_names) else str(smooth_idx)

                if smooth_label == pending_label:
                    pending_streak += 1
                else:
                    pending_label = smooth_label
                    pending_streak = 1

                if smooth_conf >= SWITCH_CONF_THRESHOLD and pending_streak >= MIN_STREAK_TO_SWITCH:
                    display_label = smooth_label
                    display_conf = smooth_conf
        else:
            sequence_buffer.clear()
            ema_probs = None
            pending_label = None
            pending_streak = 0
            display_label = "No hands"
            display_conf = 0.0

        frame_count += 1

        cv2.putText(
            frame,
            f"Stable: {display_label} ({display_conf:.2f})",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2,
        )
        cv2.putText(
            frame,
            f"Window: {len(sequence_buffer)}/{seq_len}  Streak: {pending_streak}",
            (10, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2,
        )
        cv2.putText(frame, "Press q to quit", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.imshow("MediaPipe + TFLite Real-time (Stable)", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):

            break
finally:
    cap.release()
    cv2.destroyAllWindows()